# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abhinavt1325/Flyrank-Internship-Capstone/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This notebook defines and verifies the **Search Intelligence Data Contract** for the **Content Refresh & Traffic Recovery Priority Lane** using the full FlyRank warehouse release on Hugging Face (`hf://datasets/FlyRank/internship-warehouse`).

> **Skill Loaded:** `writing-data-contracts` + `flyrank/flyrank-data`
> **Panel Reference:** Mid-panel observation partition `month=2026-03`
> **Deliverable:** 5 plain-words contract answers, 3 executed verification queries, 5-feature extraction frame, deliberate leakage trap demonstration, and data limitations.

## 1. Unit of analysis + time window

### Plain-Words Contract Answers (5 Core Specifications)
1. **Unit of Analysis (What one row means):**
   - In the **detail warehouse table** (`fact_content_daily_performance`), one row represents one **`(client_hash_id, content_hash_id, report_date)`** tuple.
   - In our **modeled feature slice**, one row represents **one pseudonymized content page (`content_hash_id`) for a specific client (`client_hash_id`)** aggregated over a defined historical observation window.
2. **Table(s) Used:**
   - `fact_content_daily_performance` (partitioned by month, iterated on mid-panel partition `month=2026-03`).
   - `dim_clients` (client access profiles and start dates for panel checks).
   - `dim_content` (metadata and content properties).
3. **Time Window:**
   - **Historical Observation Window (Features):** March 1, 2026 to March 20, 2026 (first 20 days of `month=2026-03`).
   - **Forward Outcome Window (Target/Proxy):** March 21, 2026 to March 31, 2026 (subsequent 11 days of `month=2026-03`).
4. **What to Predict or Rank (Target / Proxy):**
   - Priority score for **Observed Traffic Decline / Recovery Opportunity**: whether a content item's daily impression rate declines by more than 20% in the forward window relative to its observation window baseline (`is_declining_target = 1`), weighted by baseline traffic volume to rank high-impact decay.
5. **Deliberately Excluded:**
   - **All outcome-window metrics** (e.g. forward clicks/impressions) from feature inputs.
   - **Static product heuristic flags** (such as `trend_direction` or `trend_pct` from starter CSVs) to prevent heuristic leakage.
   - **Pseudonymous Entity IDs** (`client_hash_id`, `content_hash_id`) as direct features (reserved solely for grouping/joining and client-grouped cross-validation splits).

In [3]:
# Setup DuckDB connection and Hugging Face authentication
import os, getpass, duckdb, pandas as pd, numpy as np

# Load token from environment, .env file, Colab Secret, or interactive prompt
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN and os.path.exists('../../.env'):
    with open('../../.env') as f:
        for line in f:
            if line.startswith('HF_TOKEN='):
                HF_TOKEN = line.strip().split('=', 1)[1]
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass

con = duckdb.connect()
if HF_TOKEN:
    con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLE_MONTH_03 = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
TABLE_CLIENTS = f"read_parquet('{REL}/dim_clients.parquet')"
TABLE_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"

print("DuckDB connected to warehouse release successfully.")

DuckDB connected to warehouse release successfully.


## 2. Fields: feature / label / context / excluded

### Field Classification Table

| Bucket | Field Name | Source / Formula | Reason & Usage Rule |
|---|---|---|---|
| **Feature** | `log_obs_impressions` | `LN(1 + SUM(gsc_impressions))` [Days 1–20] | Log-scaled search visibility; fully knowable at decision day 20. |
| **Feature** | `log_obs_clicks` | `LN(1 + SUM(gsc_clicks))` [Days 1–20] | Log-scaled organic traffic; fully knowable at decision day 20. |
| **Feature** | `avg_gsc_position` | `AVG(gsc_avg_position)` on active days [Days 1–20] | Search ranking position; historical fact before decision moment. |
| **Feature** | `active_days_count` | `COUNT(CASE WHEN gsc_impressions > 0 THEN 1 END)` [Days 1–20] | Search appearance frequency (0–20 days); measured before decision moment. |
| **Feature** | `obs_ctr_pct` | `(SUM(clicks) * 100.0) / SUM(impressions)` [Days 1–20] | Click-through percentage over observation window; strictly historical. |
| **Label / Proxy** | `is_declining_target` | `(outcome_imp / 11.0) < 0.8 * (obs_imp / 20.0)` | Ground truth binary indicator of >20% daily rate drop in forward window. |
| **Context** | `client_hash_id` | `fact_content_daily_performance.client_hash_id` | Grouping key for client-holdout splits (never used as learning feature). |
| **Context** | `content_hash_id` | `fact_content_daily_performance.content_hash_id` | Unique page identifier used for entity joins and aggregation. |
| **Context** | `report_date` | `fact_content_daily_performance.report_date` | Date dimension used to enforce temporal window boundaries. |
| **Excluded** | `LEAK_future_impressions` | `SUM(gsc_impressions)` [Days 21–31] | **Future leakage:** occurs AFTER decision moment; causes artificial 100% accuracy. |
| **Excluded** | `trend_direction` / `trend_pct` | Starter CSV heuristic flags | **Label artifact:** encodes hand-written rule decisions; not an independent signal. |
| **Excluded** | Entity IDs as features | Categorical / integer ID encoding | **Memorization risk:** causes model to memorize specific clients/URLs rather than learning generalizable signals. |

In [4]:
# Inspect schema and data types of the warehouse daily performance partition
schema_df = con.sql(f"""
    DESCRIBE SELECT 
        client_hash_id, 
        content_hash_id, 
        report_date, 
        gsc_data_available, 
        ga4_data_available, 
        gsc_impressions, 
        gsc_clicks, 
        gsc_avg_position 
    FROM {TABLE_MONTH_03}
""").df()

schema_df

,column_name,column_type,null,key,default,extra
0,client_hash_id,VARCHAR,YES,None,None,None
1,content_hash_id,VARCHAR,YES,None,None,None
2,report_date,DATE,YES,None,None,None
3,gsc_data_available,BOOLEAN,YES,None,None,None
4,ga4_data_available,BOOLEAN,YES,None,None,None
5,gsc_impressions,BIGINT,YES,None,None,None
6,gsc_clicks,BIGINT,YES,None,None,None
7,gsc_avg_position,DOUBLE,YES,None,None,None


## 3. Verify it with queries (grain, counts, missing values, windows)

Here we execute the **three required contract verification queries** on the mid-panel partition `month=2026-03`:
1. **Query 1 (The Grain Probe):** Uniqueness test on `(client_hash_id, content_hash_id, report_date)` (`HAVING COUNT(*) > 1` must return exactly 0 rows).
2. **Query 2 (Slice Counts & Date Span):** Total rows, distinct entities, and exact calendar date coverage (`MIN` and `MAX` date).
3. **Query 3 (Availability & Missingness):** Verification using explicit **`IS TRUE`** filters for search (`gsc_data_available`) and analytics (`ga4_data_available`).

In [5]:
# Query 1: The Grain Probe — Prove that one row is strictly (client, content, report_date)
grain_probe_query = f"""
    SELECT 
        client_hash_id, 
        content_hash_id, 
        report_date, 
        COUNT(*) AS row_count
    FROM {TABLE_MONTH_03}
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    LIMIT 5
"""
grain_violations = con.sql(grain_probe_query).df()
print(f"Grain Violation Check: {len(grain_violations)} violations found (0 expected).")
grain_violations

Grain Violation Check: 0 violations found (0 expected).


,client_hash_id,content_hash_id,report_date,row_count


In [6]:
# Query 2: Slice Row Count and Date Span
count_span_query = f"""
    SELECT 
        COUNT(*) AS total_rows,
        COUNT(DISTINCT client_hash_id) AS distinct_clients,
        COUNT(DISTINCT content_hash_id) AS distinct_content_items,
        MIN(report_date) AS min_report_date,
        MAX(report_date) AS max_report_date
    FROM {TABLE_MONTH_03}
"""
df_counts = con.sql(count_span_query).df()
df_counts

,total_rows,distinct_clients,distinct_content_items,min_report_date,max_report_date
0,9841378,55,331437,2026-03-01,2026-03-31


In [7]:
# Query 3: Availability Verification with IS TRUE
# Note: Flags can be TRUE, FALSE, or NULL — 'IS TRUE' is mandatory to avoid silent mishandling.
availability_query = f"""
    SELECT 
        COUNT(*) AS total_slice_rows,
        COUNT(CASE WHEN gsc_data_available IS TRUE THEN 1 END) AS gsc_available_rows,
        COUNT(CASE WHEN ga4_data_available IS TRUE THEN 1 END) AS ga4_available_rows,
        COUNT(CASE WHEN gsc_data_available IS TRUE AND ga4_data_available IS TRUE THEN 1 END) AS both_available_rows,
        ROUND(COUNT(CASE WHEN gsc_data_available IS TRUE THEN 1 END) * 100.0 / COUNT(*), 2) AS gsc_available_pct,
        ROUND(COUNT(CASE WHEN ga4_data_available IS TRUE THEN 1 END) * 100.0 / COUNT(*), 2) AS ga4_available_pct,
        ROUND(COUNT(CASE WHEN gsc_data_available IS TRUE AND ga4_data_available IS TRUE THEN 1 END) * 100.0 / COUNT(*), 2) AS both_available_pct
    FROM {TABLE_MONTH_03}
"""
df_avail = con.sql(availability_query).df()
df_avail

,total_slice_rows,gsc_available_rows,ga4_available_rows,both_available_rows,gsc_available_pct,ga4_available_pct,both_available_pct
0,9841378,3611061,413966,364347,36.69,4.21,3.7


### Five Features (Max 5) + "Available When?" Proof

Below we extract the feature matrix inside DuckDB SQL and bring only the aggregated per-content dataframe into pandas:

1. **`log_obs_impressions`**: `LN(1 + SUM(gsc_impressions))` over Days 1–20.
   - *Knowable at decision moment because past 20-day search impressions have already been recorded by Google Search Console before the refresh decision is made.*
2. **`log_obs_clicks`**: `LN(1 + SUM(gsc_clicks))` over Days 1–20.
   - *Knowable at decision moment because organic clicks earned during Days 1–20 are historical logged events.*
3. **`avg_gsc_position`**: `AVG(gsc_avg_position)` across active days in Days 1–20.
   - *Knowable at decision moment because search ranking positions during the observation window are established historical facts.*
4. **`active_days_count`**: Number of days with $\ge 1$ impression in Days 1–20 (0–20).
   - *Knowable at decision moment because daily impression consistency across the past 20 days is fully measured.*
5. **`obs_ctr_pct`**: $(\text{clicks} \times 100.0) / \text{impressions}$ over Days 1–20.
   - *Knowable at decision moment because it is computed exclusively from historical observation-window metrics.*

In [8]:
# Build the 5-feature matrix and extract the deliberate leak column inside DuckDB SQL
feature_sql = f"""
    WITH daily_agg AS (
        SELECT 
            client_hash_id,
            content_hash_id,
            -- Observation Window (Days 1–20 of March 2026)
            SUM(CASE WHEN DAY(report_date) <= 20 THEN gsc_impressions ELSE 0 END) AS obs_impressions,
            SUM(CASE WHEN DAY(report_date) <= 20 THEN gsc_clicks ELSE 0 END) AS obs_clicks,
            AVG(CASE WHEN DAY(report_date) <= 20 AND gsc_impressions > 0 THEN gsc_avg_position END) AS obs_avg_position,
            COUNT(CASE WHEN DAY(report_date) <= 20 AND gsc_impressions > 0 THEN 1 END) AS obs_active_days,
            
            -- Outcome Window (Days 21–31 of March 2026)
            SUM(CASE WHEN DAY(report_date) > 20 THEN gsc_impressions ELSE 0 END) AS outcome_impressions,
            SUM(CASE WHEN DAY(report_date) > 20 THEN gsc_clicks ELSE 0 END) AS outcome_clicks
        FROM {TABLE_MONTH_03}
        WHERE gsc_data_available IS TRUE
        GROUP BY 1, 2
        HAVING SUM(CASE WHEN DAY(report_date) <= 20 THEN gsc_impressions ELSE 0 END) >= 30
    )
    SELECT 
        client_hash_id,
        content_hash_id,
        -- 5 Honest Features:
        LN(1 + obs_impressions) AS log_obs_impressions,
        LN(1 + obs_clicks) AS log_obs_clicks,
        COALESCE(obs_avg_position, 100.0) AS avg_gsc_position,
        obs_active_days AS active_days_count,
        (obs_clicks * 100.0 / NULLIF(obs_impressions, 0)) AS obs_ctr_pct,
        
        -- Target Ground Truth (>20% drop in daily impression rate):
        CASE WHEN (outcome_impressions / 11.0) < 0.8 * (obs_impressions / 20.0) THEN 1 ELSE 0 END AS is_declining_target,
        
        -- THE TRAP: Deliberate label-leaked column (future outcome data)
        outcome_impressions AS LEAK_future_impressions
    FROM daily_agg
"""

df_features = con.sql(feature_sql).df()
print(f"Extracted feature dataset: {df_features.shape[0]:,} content items x {df_features.shape[1]} columns.")
print("\nTarget distribution:")
print(df_features['is_declining_target'].value_counts(normalize=True).round(4))
df_features.head()

Extracted feature dataset: 112,315 content items x 9 columns.

Target distribution:
is_declining_target
0    0.6726
1    0.3274
Name: proportion, dtype: float64


,client_hash_id,content_hash_id,log_obs_impressions,log_obs_clicks,avg_gsc_position,active_days_count,obs_ctr_pct,is_declining_target,LEAK_future_impressions
0,client_e547b89c05043229,content_498a9afb9da07372,7.232010,0.000000,4.728619,18,0.000000,1,512.0
1,client_e547b89c05043229,content_32185762106c6e3e,7.924434,2.772589,4.711236,18,0.542888,0,2359.0
2,client_e547b89c05043229,content_98279f3863475491,6.569481,0.693147,17.225477,18,0.140449,0,406.0
3,client_e547b89c05043229,content_6b99af0551ff7a09,8.215547,0.693147,44.478123,18,0.027049,0,2830.0
4,client_e547b89c05043229,content_2a9cabd2c0a98312,6.210600,0.000000,17.091431,18,0.000000,0,303.0


### The Trap: Springing and Removing the Leakage Trap

We now perform the leakage experiment:
1. **Honest Model:** Train a classifier using strictly our 5 honest features.
2. **Leaked Model:** Add `LEAK_future_impressions` (a column derived from the outcome window). Watch the metrics artificially skyrocket toward near-perfect accuracy.
3. **Remediation:** Remove the leaked column and preserve the honest model performance.

In [9]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score

# Define honest feature set vs leaked feature set
honest_features = ['log_obs_impressions', 'log_obs_clicks', 'avg_gsc_position', 'active_days_count', 'obs_ctr_pct']
leaked_features = honest_features + ['LEAK_future_impressions']

X_honest = df_features[honest_features]
X_leaked = df_features[leaked_features]
y = df_features['is_declining_target']

# Stratified train/test split
X_tr_h, X_te_h, y_tr, y_te = train_test_split(X_honest, y, test_size=0.25, random_state=42, stratify=y)
X_tr_l, X_te_l, _, _ = train_test_split(X_leaked, y, test_size=0.25, random_state=42, stratify=y)

# 1. Fit Honest Model
clf_honest = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42, n_jobs=-1)
clf_honest.fit(X_tr_h, y_tr)
probs_honest = clf_honest.predict_proba(X_te_h)[:, 1]
preds_honest = clf_honest.predict(X_te_h)
roc_honest = roc_auc_score(y_te, probs_honest)

# 2. Fit Leaked Model (The Trap)
clf_leaked = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42, n_jobs=-1)
clf_leaked.fit(X_tr_l, y_tr)
probs_leaked = clf_leaked.predict_proba(X_te_l)[:, 1]
preds_leaked = clf_leaked.predict(X_te_l)
roc_leaked = roc_auc_score(y_te, probs_leaked)

print("===========================================================")
print(f"HONEST MODEL ROC-AUC (5 Features):                 {roc_honest:.4f}")
print(f"LEAKED MODEL ROC-AUC (5 Features + Leaked Future): {roc_leaked:.4f}")
print("===========================================================")

print("\n--- Honest Model Classification Report ---")
print(classification_report(y_te, preds_honest, digits=4))

print("--- Leaked Model Classification Report (THE TRAP) ---")
print(classification_report(y_te, preds_leaked, digits=4))

# 3. Delete the leaked column and preserve honest feature matrix
df_features.drop(columns=['LEAK_future_impressions'], inplace=True)
print("Leakage column 'LEAK_future_impressions' permanently dropped from feature dataset.")

HONEST MODEL ROC-AUC (5 Features):                 0.6571
LEAKED MODEL ROC-AUC (5 Features + Leaked Future): 0.9292

--- Honest Model Classification Report ---
              precision    recall  f1-score   support

           0     0.6748    0.9965    0.8047     18886
           1     0.6489    0.0133    0.0260      9193

    accuracy                         0.6746     28079
   macro avg     0.6619    0.5049    0.4153     28079
weighted avg     0.6663    0.6746    0.5497     28079

--- Leaked Model Classification Report (THE TRAP) ---
              precision    recall  f1-score   support

           0     0.7878    0.9981    0.8806     18886
           1     0.9916    0.4476    0.6168      9193

    accuracy                         0.8179     28079
   macro avg     0.8897    0.7229    0.7487     28079
weighted avg     0.8545    0.8179    0.7942     28079

Leakage column 'LEAK_future_impressions' permanently dropped from feature dataset.


## 4. Data limits

### Named Limitations of This Slice
1. **Unbalanced Panel & Asymmetric Service Onboarding:**
   - Query 3 shows that in `month=2026-03`, only **36.69%** of rows have `gsc_data_available IS TRUE` and only **4.21%** have `ga4_data_available IS TRUE`. Many clients onboarded late or do not have active GA4 tracking. Filtering with `IS TRUE` is mandatory; rows with `ga4_data_available = FALSE` or `NULL` must not be treated as zero engagement.
2. **Zero-Impression Days & Average Rank Bias:**
   - On days when a content item receives 0 search impressions, Google Search Console does not log an `avg_position` (it appears as NULL or 0 in raw logs). A naive average across all days severely biases ranking metrics downward. Position metrics must strictly be conditioned on `gsc_impressions > 0`.
3. **Fixed Observation Window Boundary Effects:**
   - Fixed calendar windows (e.g. 20 days) create measurement noise for low-volume tail pages. Applying a volume floor (`obs_impressions >= 30`) ensures stable denominator scaling and prevents low-volume noise from dominating ranking scores.

In [10]:
# Code check: Verify client-level variation in GSC and GA4 availability across dim_clients
client_profile_query = f"""
    SELECT 
        access_profile,
        COUNT(*) AS client_count,
        COUNT(CASE WHEN gsc_data_start IS NOT NULL THEN 1 END) AS clients_with_gsc,
        COUNT(CASE WHEN ga4_data_start IS NOT NULL THEN 1 END) AS clients_with_ga4
    FROM {TABLE_CLIENTS}
    GROUP BY access_profile
    ORDER BY client_count DESC
"""
df_client_profiles = con.sql(client_profile_query).df()
df_client_profiles

,access_profile,client_count,clients_with_gsc,clients_with_ga4
0,gsc_and_ga4,53,50,50
1,no_search_or_analytics_access,26,3,0
2,gsc_only,14,10,0
3,source_only_missing_client_dimension,10,4,1
4,ga4_only,1,0,0


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.